# AI Revenue Recovery — EDA & Model Training

Razorpay AI Buildathon 2026 — Revenue Recovery track.

This notebook:
1. Loads the synthetic, leakage-safe transaction dataset
2. Explores failure patterns
3. Trains a **failure-reason classifier** (multiclass)
4. Trains a **retry-success predictor** (binary)
5. Saves both as `.pkl` files for use in the API

**Leakage safeguards used throughout:**
- Train/test split is by `customer_id`, done at data-generation time — no customer appears in both sets
- All encoders/preprocessing are fit only on train, applied to test
- Models never see `retry_success` or `failed` as an input feature for the wrong task


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 1. Load data (pre-split by customer, not by row)

In [ ]:
train_df = pd.read_csv("../data/synthetic/transactions_train.csv")
test_df = pd.read_csv("../data/synthetic/transactions_test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head()


In [ ]:
# Sanity check: confirm NO customer overlap between train and test (leakage check)
overlap = set(train_df.customer_id) & set(test_df.customer_id)
print(f"Customer overlap between train/test: {len(overlap)} (should be 0)")


## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_df["failed"].value_counts(normalize=True).plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Overall failure rate (train)")
axes[0].set_xticklabels(["Success", "Failed"], rotation=0)

train_df[train_df.failed == 1]["failure_reason"].value_counts().plot(
    kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Failure reason breakdown")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=train_df, x="failed", y="network_latency_ms", ax=axes[0])
axes[0].set_title("Network latency vs failure")

sns.countplot(data=train_df, x="payment_method", hue="failed", ax=axes[1])
axes[1].set_title("Failure by payment method")
plt.tight_layout()
plt.show()


## 3. Feature / target setup

`failure_reason` is only defined for failed transactions, so both models below
train only on the failed subset (matching the real-world scenario: you only
run failure-classification + retry-optimization once you already know a
payment failed).


In [ ]:
CATEGORICAL = ["payment_method", "bank", "card_type", "device_type"]
NUMERIC = [
    "amount", "hour_of_day", "day_of_week", "is_weekend",
    "customer_tenure_days", "prior_failed_attempts_30d",
    "network_latency_ms", "retry_attempt_number",
]

train_fail = train_df[train_df.failed == 1].copy()
test_fail = test_df[test_df.failed == 1].copy()

print(f"Failed rows -> train: {len(train_fail)}, test: {len(test_fail)}")


## 4. Model 1 — Failure Reason Classifier (multiclass)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, ConfusionMatrixDisplay

X_train, y_train = train_fail[CATEGORICAL + NUMERIC], train_fail["failure_reason"]
X_test, y_test = test_fail[CATEGORICAL + NUMERIC], test_fail["failure_reason"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("num", "passthrough", NUMERIC),
])

failure_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1)),
])

# IMPORTANT: fit only on train
failure_pipe.fit(X_train, y_train)


In [ ]:
preds = failure_pipe.predict(X_test)  # evaluated only on held-out test split
acc = accuracy_score(y_test, preds)
macro_f1 = f1_score(y_test, preds, average="macro")

print(f"Test accuracy: {acc:.4f}")
print(f"Macro F1:      {macro_f1:.4f}\n")
print(classification_report(y_test, preds))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, preds, xticks_rotation=45, ax=ax, colorbar=False)
plt.title("Failure Reason — Confusion Matrix (test set)")
plt.tight_layout()
plt.show()


## 5. Model 2 — Retry Success Predictor (binary)

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

features_retry = CATEGORICAL + NUMERIC + ["failure_reason"]
X_train_r = train_fail[features_retry]
y_train_r = train_fail["retry_success"].astype(int)
X_test_r = test_fail[features_retry]
y_test_r = test_fail["retry_success"].astype(int)

preprocessor_r = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL + ["failure_reason"]),
    ("num", "passthrough", NUMERIC),
])

retry_pipe = Pipeline([
    ("prep", preprocessor_r),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1)),
])

retry_pipe.fit(X_train_r, y_train_r)


In [ ]:
preds_r = retry_pipe.predict(X_test_r)
proba_r = retry_pipe.predict_proba(X_test_r)[:, 1]

acc_r = accuracy_score(y_test_r, preds_r)
auc_r = roc_auc_score(y_test_r, proba_r)

print(f"Test accuracy: {acc_r:.4f}")
print(f"ROC-AUC:       {auc_r:.4f}")

RocCurveDisplay.from_predictions(y_test_r, proba_r)
plt.title("Retry Success — ROC Curve (test set)")
plt.show()


## 6. Feature Importance (explainability for the pitch)

In [ ]:
importances = failure_pipe.named_steps["clf"].feature_importances_
feature_names = failure_pipe.named_steps["prep"].get_feature_names_out()

imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df, x="importance", y="feature", color="#4C72B0")
plt.title("Top 15 features — Failure Reason model")
plt.tight_layout()
plt.show()


## 7. Save models as .pkl for the API / demo

In [ ]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(failure_pipe, MODEL_DIR / "failure_model.pkl")
joblib.dump(retry_pipe, MODEL_DIR / "retry_success_model.pkl")

with open(MODEL_DIR / "failure_model_metrics.json", "w") as f:
    json.dump({
        "accuracy": acc, "macro_f1": macro_f1,
        "report": classification_report(y_test, preds, output_dict=True),
    }, f, indent=2)

with open(MODEL_DIR / "retry_success_model_metrics.json", "w") as f:
    json.dump({"accuracy": acc_r, "roc_auc": auc_r}, f, indent=2)

print("Saved failure_model.pkl and retry_success_model.pkl to", MODEL_DIR.resolve())


## 8. Quick inference sanity check

In [ ]:
sample = pd.DataFrame([{
    "payment_method": "netbanking", "bank": "IDFC", "card_type": "debit",
    "device_type": "mobile", "amount": 1200, "hour_of_day": 3,
    "day_of_week": 2, "is_weekend": 0, "customer_tenure_days": 400,
    "prior_failed_attempts_30d": 0, "network_latency_ms": 300,
    "retry_attempt_number": 1,
}])

predicted_reason = failure_pipe.predict(sample)[0]
print("Predicted failure reason:", predicted_reason)

sample["failure_reason"] = predicted_reason
retry_prob = retry_pipe.predict_proba(sample[features_retry])[0][1]
print(f"Predicted retry success probability: {retry_prob:.2%}")
